In [ ]:
%pip install typing_extensions git+https://github.com/AgDMALabs-Public/ag-vision-dataops.git

In [ ]:
import pandas as pd
import os
from pyspark.sql import SparkSession

# Assumes you have a SparkSession named 'spark' available
spark = SparkSession.builder.getOrCreate()

In [ ]:
from ag_vision.pipelines import image_processing as ip

In [ ]:
TABLE_NAME = "use1_prod_artemis_catalog_3718194974443840.production.images_table"

In [ ]:
imgs_df = spark.table(TABLE_NAME)

In [ ]:
def process_batch(iterator):
    """
    Processes an iterator of pandas DataFrames (chunks) and yields processed results.
    """
    # Initialize models once per partition
    # Define a local path for the model cache
    # /tmp/model_cache is usually safe on Databricks/Spark nodes
    cache_dir = "/tmp/hf_model_cache"
    os.makedirs(cache_dir, exist_ok=True)
    lock_path = os.path.join(cache_dir, "model_loading.lock")

    for pdf in iterator:
        # pdf is a pandas DataFrame chunk
        items = pdf["file_path"].tolist()

        if not items:
            # Yield an empty DataFrame with the correct columns if the chunk is empty
            yield pd.DataFrame(columns=['file_path', 'status'])
        else:
            # Process the items and yield the resulting DataFrame
            yield ip.generate_metadata_files_from_image_list(file_paths=items,
                                                             platform='db',
                                                             image_type='original',
                                                             model_cache_dir=cache_dir)


In [ ]:
if len(imgs_df):
    img_spark_df = imgs_df.mapInPandas(process_batch,
                                       schema="file_path string, status string, be string, me string")

    status_df = img_spark_df.toPandas()
    print(status_df)